# Notebook 06: XGBoost Depth Comparison

This notebook continues the XGBoost tuning track in the updated series. The saved output names still follow the original run label so that the notebook stays aligned with earlier artifacts.

In [1]:
# 05_xgboost_test_2.ipynb
# Experiment:XGBoost 500 trees + max_depth tuning

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, recall_score, f1_score
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("05_xgboost_test_2: 500 trees + max_depth tuning") 
print("="*60)

# 1. Load data
print("\n1. Load data...") 
train = pd.read_csv('../data/train.csv')
print(f"Train shape: {train.shape}") 

# 2. Prepare data
print("\n2. Prepare data...") 
X = train.drop(['id', 'diagnosed_diabetes'], axis=1)
y = train['diagnosed_diabetes']

# Encode categorical features
categorical_cols = ['gender', 'ethnicity', 'education_level', 
                   'income_level', 'smoking_status', 'employment_status']
X_encoded = pd.get_dummies(X, columns=categorical_cols)
print(f"Encoded feature count: {X_encoded.shape[1]}") 

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train set: {X_train.shape}, Validation set: {X_val.shape}") 

# 3. Compute class weights
n_negative = sum(y_train == 0)
n_positive = sum(y_train == 1)
scale_pos_weight = n_negative / n_positive
print(f"\nscale_pos_weight: {scale_pos_weight:.3f}")

# 4. Previous experiment results
prev_exp = {
    'name': '04_xgboost_test_1 (300 trees)',
    'auc': 0.7255,
    'recall_0': 0.695,
    'recall_1': 0.631,
    'params': {'n_estimators': 300, 'max_depth': 6}
}

print(f"\nPrevious experiment [{prev_exp['name']}] results:") 
print(f"- AUC: {prev_exp['auc']}")
print(f"- Recall class 0: {prev_exp['recall_0']}") 
print(f"- Recall class 1: {prev_exp['recall_1']}") 
print(f"- Recall gap: {prev_exp['recall_1'] - prev_exp['recall_0']:.3f}") 

# 5. Try different max_depth values
print("\n5. Try different max_depth values...") 

depths_to_try = [4, 6, 8, 10]
results_dict = {}

plt.figure(figsize=(15, 10))

for idx, depth in enumerate(depths_to_try):
    print(f"\n▶ Train max_depth={depth}...") 
    
    model = xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.1,
        max_depth=depth,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        eval_metric='auc',
        use_label_encoder=False,
        early_stopping_rounds=50
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    # Evaluation
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    y_pred = model.predict(X_val)
    
    auc = roc_auc_score(y_val, y_pred_proba)
    recall_0 = recall_score(y_val, y_pred, pos_label=0)
    recall_1 = recall_score(y_val, y_pred, pos_label=1)
    f1_0 = f1_score(y_val, y_pred, pos_label=0)
    f1_1 = f1_score(y_val, y_pred, pos_label=1)
    
    results_dict[depth] = {
        'model': model,
        'auc': auc,
        'recall_0': recall_0,
        'recall_1': recall_1,
        'f1_0': f1_0,
        'f1_1': f1_1,
        'results': model.evals_result()
    }
    
    print(f" AUC: {auc:.4f} | Recall class 0: {recall_0:.3f} | Recall class 1: {recall_1:.3f} | Mean F1: {(f1_0+f1_1)/2:.3f}") 
    
    # Plot learning curves(subplot)
    plt.subplot(2, 2, idx+1)
    results = model.evals_result()
    plt.plot(results['validation_0']['auc'], label=f'max_depth={depth}')
    plt.axhline(y=prev_exp['auc'], color='gray', linestyle='--', alpha=0.5, label='Previous best')
    plt.xlabel('Rounds')
    plt.ylabel('AUC')
    plt.title(f'Learning Curve (max_depth={depth})')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../logs/05_xgboost_test_2_learning_curves.png', dpi=100, bbox_inches='tight')
plt.show()

# 6. Find the best depth
print("\n6. Result comparison...") 

best_depth = max(results_dict, key=lambda d: results_dict[d]['auc'])
best_model = results_dict[best_depth]['model']
best_auc = results_dict[best_depth]['auc']
best_recall_0 = results_dict[best_depth]['recall_0']
best_recall_1 = results_dict[best_depth]['recall_1']

print("\nAll max_depth results:") 
print("-" * 70)
print(f"{'depth':<8} {'AUC':<8} {'Recall_0':<10} {'Recall_1':<10} {'F1_0':<8} {'F1_1':<8} {'F1_avg':<8}")
print("-" * 70)
for depth in depths_to_try:
    r = results_dict[depth]
    f1_avg = (r['f1_0'] + r['f1_1']) / 2
    print(f"{depth:<8} {r['auc']:<8.4f} {r['recall_0']:<10.3f} {r['recall_1']:<10.3f} {r['f1_0']:<8.3f} {r['f1_1']:<8.3f} {f1_avg:<8.3f}")
print("-" * 70)
print(f"Best depth: {best_depth} (AUC={best_auc:.4f})") 

# 7. Compare with the previous experiment
print(f"\n7. Compare with the previous experiment:") 
print(f"- Previous ({prev_exp['name']}): AUC={prev_exp['auc']:.4f}, Recall class 0={prev_exp['recall_0']:.3f}, Recall class 1={prev_exp['recall_1']:.3f}") 
print(f"- Current (depth={best_depth}): AUC={best_auc:.4f}, Recall class 0={best_recall_0:.3f}, Recall class 1={best_recall_1:.3f}") 
print(f"- AUC improvement: +{best_auc - prev_exp['auc']:.4f}") 

# 8. Threshold optimization analysis(using the best model)
print("\n8. Threshold optimization analysis...") 

y_pred_proba_best = best_model.predict_proba(X_val)[:, 1]

thresholds = [0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7]
print("\nPerformance under different thresholds:") 
print("-" * 75)
print(f"{'Threshold':<8} {'Recall class 0':<10} {'Recall class 1':<10} {'gap':<8} {'F1_0':<8} {'F1_1':<8} {'F1_avg':<8}") 
print("-" * 75)

best_f1_avg = 0
best_threshold = 0.5

for thresh in thresholds:
    y_pred_thresh = (y_pred_proba_best >= thresh).astype(float)
    r0 = recall_score(y_val, y_pred_thresh, pos_label=0)
    r1 = recall_score(y_val, y_pred_thresh, pos_label=1)
    f1_0 = f1_score(y_val, y_pred_thresh, pos_label=0)
    f1_1 = f1_score(y_val, y_pred_thresh, pos_label=1)
    f1_avg = (f1_0 + f1_1) / 2
    gap = r1 - r0
    print(f"{thresh:<8.2f} {r0:<10.3f} {r1:<10.3f} {gap:<8.3f} {f1_0:<8.3f} {f1_1:<8.3f} {f1_avg:<8.3f}")
    
    if f1_avg > best_f1_avg:
        best_f1_avg = f1_avg
        best_threshold = thresh

print("-" * 75)
print(f"Best threshold: {best_threshold} (Mean F1={best_f1_avg:.3f})") 

# 9. Feature importanceVisualization
print("\n9. Feature importance analysis...") 

plt.figure(figsize=(12, 8))
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.barh(importance['feature'], importance['importance'])
plt.xlabel('Importance')
plt.title(f'Top 20 Feature Importance (max_depth={best_depth})')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../logs/05_xgboost_test_2_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nTop 10 most important features:") 
print(importance.head(10).to_string(index=False))

# 10. Save model and files
print("\n10. Save model and submission file...") 

model_path = f'../models/05_xgboost_test_2_depth{best_depth}.pkl'
joblib.dump(best_model, model_path)
print(f"Model saved to {model_path}") 

test = pd.read_csv('../data/test.csv')
X_test = test.drop(['id'], axis=1)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols)

missing_cols = set(X_train.columns) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0
X_test_encoded = X_test_encoded[X_train.columns]

test_pred = best_model.predict_proba(X_test_encoded)[:, 1]

submission = pd.DataFrame({
    'id': test['id'],
    'diagnosed_diabetes': test_pred
})
submission_path = '../submissions/05_xgboost_test_2.csv'
submission.to_csv(submission_path, index=False)
print(f"Submission file saved to {submission_path}")


05_xgboost_test_2: 500 trees + max_depth tuning ============================================================

1. Load data... Train shape: (700000, 26) 
2. Prepare data... Encoded feature count: 42 Train set: (560000, 42), Validation set: (140000, 42) 
scale_pos_weight: 0.604

Previous experiment [04_xgboost_test_1 (300 trees)] results: - AUC: 0.7255
- Recall class 0: 0.695 - Recall class 1: 0.631 - Recall gap: -0.064 
5. Try different max_depth values... 
▶ Train max_depth=4...  AUC: 0.7257 | Recall class 0: 0.698 | Recall class 1: 0.628 | Mean F1: 0.648 
▶ Train max_depth=6...  AUC: 0.7264 | Recall class 0: 0.692 | Recall class 1: 0.635 | Mean F1: 0.650 
▶ Train max_depth=8...  AUC: 0.7253 | Recall class 0: 0.687 | Recall class 1: 0.638 | Mean F1: 0.650 
▶ Train max_depth=10...  AUC: 0.7228 | Recall class 0: 0.681 | Recall class 1: 0.640 | Mean F1: 0.648 

<Figure size 1500x1000 with 4 Axes>


6. Result comparison... 
All max_depth results: ----------------------------------------------------------------------
depth    AUC      Recall_0   Recall_1   F1_0     F1_1     F1_avg  
----------------------------------------------------------------------
4        0.7257   0.698      0.628      0.603    0.693    0.648   
6        0.7264   0.692      0.635      0.603    0.698    0.650   
8        0.7253   0.687      0.638      0.601    0.699    0.650   
10       0.7228   0.681      0.640      0.598    0.698    0.648   
----------------------------------------------------------------------
Best depth: 6 (AUC=0.7264) 
7. Compare with the previous experiment: - Previous (04_xgboost_test_1 (300 trees)): AUC=0.7255, Recall class 0=0.695, Recall class 1=0.631 - Current (depth=6): AUC=0.7264, Recall class 0=0.692, Recall class 1=0.635 - AUC improvement: +0.0009 
8. Threshold optimization analysis... 
Performance under different thresholds: ----------------------------------------------------

<Figure size 1200x800 with 1 Axes>


Top 10 most important features: feature  importance
           family_history_diabetes    0.766882
                               age    0.031437
physical_activity_minutes_per_week    0.029840
                     triglycerides    0.010445
                               bmi    0.008610
                   ldl_cholesterol    0.007310
            cardiovascular_history    0.006974
                        diet_score    0.005428
                   hdl_cholesterol    0.005418
                        heart_rate    0.005087

10. Save model and submission file... Model saved to ../models/05_xgboost_test_2_depth6.pkl Submission file saved to ../submissions/05_xgboost_test_2.csv 

# Experiment 6 Summary: XGBoost Depth Comparison at 500 Trees

## Role in the series
This stage studies whether a larger tree budget and a controlled depth comparison can push the boosting baseline further. It continues the XGBoost branch rather than introducing a new model family.

## Interpretation
The notebook shows that additional capacity can improve AUC, but the gain needs to be judged carefully against model complexity. A practical reading is that stronger results are still available within XGBoost, which justifies continuing the tuning sequence.

## Reproducibility note
The code cell and saved outputs retain their original run label for consistency with previously generated artifacts.